# HAKE-MER — M1+M2+M3+M4 (NRC, H4)

**Modules:** M1 + M2 + M3 (NRC document prior) + M4 · DistilBERT-base

**Protocol:** batch 16, LR 5e-5, 4 epochs, 3 seeds.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
e50c957


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.3 MB/s eta 0:00:00


In [4]:
!./run_m1_m2_m3_m4_campaign.sh --lexicon nrc --backbone distilbert-base-uncased {TRAIN_FLAGS}

config.json: 100% 483/483 [00:00<00:00, 4.70MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 549kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 8.97MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 20.4MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 33.7MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:  48% 1.34M/2.77M [00:00<00:00, 4.32MB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 5.97MB/s,  268kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 6.05MB/s,  273kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:  57% 199k/350k [00:00<00:00, 883kB/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:00<00:00, 1.46MB/s, 34.4kB/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing file: 100% 350k/350k [00:00<00:00, 1.48MB/s, 34.9kB/s  ]

simplified/test-00000-of-00001.parquet: downloading bytes:   0% 0.

In [5]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_m2_m3_nrc_m4_distilbert_base_uncased_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

  f1_micro: 0.5777 ± 0.0038
  f1_macro: 0.5099 ± 0.0028
  exact_match: 0.4536 ± 0.0072
  map: 0.5129 ± 0.0007


In [6]:
import zipfile
from google.colab import files

slug = "distilbert_base_uncased"
campaign = Path(f"reference/artifacts/m1_m2_m3_nrc_m4_{slug}_campaign.json")
out_name = "m1_m2_m3_nrc_m4_distilbert_step0.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_m3_nrc_m4/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name}")
files.download(str(zip_path))

Download m1_m2_m3_nrc_m4_distilbert_step0.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Archive

Executed **.ipynb** + zip → back to repo / agent for H4 NRC row in the report.